In [7]:
import json
from riotwatcher import LolWatcher, ApiError
from riotwatcher._apis.riot import AccountApi
import pandas as pd

def get_last_match_id(watcher, region, puuid):
    """Returns the most recent match ID for the given PUUID."""
    match_ids = watcher.match.matchlist_by_puuid(region, puuid, count=1)
    return match_ids[0] if match_ids else None


def get_match_details(watcher, region, match_id):
    return watcher.match.by_id(region, match_id)


def get_match_ids_by_puuid(watcher, region, puuid, count):
    return watcher.match.matchlist_by_puuid(region, puuid, count=count + 1)


def extract_puuids(match_data):
    try:
        return match_data["metadata"]["participants"]
    except Exception:
        return []



In [ ]:

def main():
    # Paramètres codés en dur ici (à modifier selon besoin)
    api_key = ''  # Mets ta clé API ici
    region = 'asia'  # Pour les serveurs coréens (KR = asia region)
    platform = 'kr'  # plateforme API
    name = 'Sard'    # Riot gameName
    tag = 'CASS'     # Riot tagLine
    count = 2        # nombre de matchs récents à récupérer
    save = True      # sauvegarder le résultat en JSON
    stdout = True    # afficher le résultat et sauvegarder dans test.json

    watcher = LolWatcher(api_key)
    account_api = AccountApi(watcher._base_api)

    # Résolution du PUUID depuis name + tag
    try:
        account = account_api.by_riot_id(region, name, tag)
        puuid = account.get("puuid")
        print(f"Resolved PUUID: {puuid}")
    except ApiError as e:
        print(f"Error fetching account by name/tag: {e}")
        return

    if not puuid:
        print("PUUID not found.")
        return

    # Recup du last match pour cet uid
    try:
        last_match_id = get_last_match_id(watcher, region, puuid)
    except ApiError as e:
        print(f"Error fetching match list: {e}")
        return

    if not last_match_id:
        print("No matches found for this account.")
        return

    print(f"Latest match: {last_match_id}")

    try:
        match = get_match_details(watcher, region, last_match_id)
    except ApiError as e:
        print(f"Error fetching match: {e}")
        return

    # Recup les uids des autres joueurs de la game
    puuids = extract_puuids(match)
    if not puuids:
        print("Failed to extract participant PUUIDs.")
        return

    result = {"latest_match": last_match_id, "players": {}}

    # On passe dans les 2 dernieres games de chaque joueurs et on recup les stats
    for p in puuids:
        try:
            ids = get_match_ids_by_puuid(watcher, region, p, count) or []
        except ApiError:
            ids = []

        filtered = [mid for mid in ids if mid != last_match_id]
        chosen = filtered[:count]

        entry = {"puuid": p, "recent_match_ids": chosen, "matches": []}
        for mid in chosen:
            try:
                md = get_match_details(watcher, region, mid)
                entry["matches"].append({"match_id": mid, "data": md})
            except ApiError as e:
                entry["matches"].append({"match_id": mid, "error": str(e)})

        result["players"][p] = entry

    if save:
        filename = f"recent_from_{puuid[:8]}_{count}.json"
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        print(f"Saved to {filename}")

    if stdout:
        with open("test.json", "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        print("Résultat sauvegardé dans test.json")

    # Création du tableau pandas avec kills, deaths, assists
    rows = []

    for puuid_key, data in result["players"].items():
        for match in data["matches"]:
            if "data" in match:
                participants = match["data"].get("info", {}).get("participants", [])
                participant_data = next((p for p in participants if p.get("puuid") == puuid_key), None)
                if participant_data:
                    rows.append({
                        "puuid": puuid_key,
                        "match_id": match["match_id"],
                        "kills": participant_data.get("kills", 0),
                        "deaths": participant_data.get("deaths", 0),
                        "assists": participant_data.get("assists", 0),
                    })

    df = pd.DataFrame(rows)
    return df


In [8]:
df = main()

Resolved PUUID: QFvZZppyrmPXfyfOuhoBAwdKlHs7FNuEW5hIdRjMxypBt3aZUPjl9bMsgQtA2gVJxXSAaZpMJbrO6w
Latest match: KR_7915260113
Saved to recent_from_QFvZZppy_2.json
Résultat sauvegardé dans test.json


In [9]:
df

,puuid,match_id,kills,deaths,assists
0,bmN4VqM7GexKbYk4IkJ-bEjui42wmXAgLbGyItIl6pyWZB...,KR_7912638966,7,5,11
1,bmN4VqM7GexKbYk4IkJ-bEjui42wmXAgLbGyItIl6pyWZB...,KR_7912594984,3,5,0
2,TUEYfx8YYUw-yS5sv09gpSQEGjASoSjzTRUkeLLjKAgnhG...,KR_7915170307,13,6,9
3,TUEYfx8YYUw-yS5sv09gpSQEGjASoSjzTRUkeLLjKAgnhG...,KR_7914484376,2,8,3
4,S1jintlXDQuozTbF2eECanCaz_GH8hnPeUJDxhjIkKC4aj...,KR_7911722657,2,10,2
5,S1jintlXDQuozTbF2eECanCaz_GH8hnPeUJDxhjIkKC4aj...,KR_7911694919,5,5,0
6,TOPz_LnLh2jEcTBTYA-qwwK3-JHeDAGdWbc8qQPdbUvdzy...,KR_7915189257,4,5,4
7,TOPz_LnLh2jEcTBTYA-qwwK3-JHeDAGdWbc8qQPdbUvdzy...,KR_7915110217,6,10,4
8,Lrzfdzb3Qe7rGJhEFzTDANjE0oHvMRp5SFiJxUE6aU2eLo...,KR_7915221195,1,2,7
9,Lrzfdzb3Qe7rGJhEFzTDANjE0oHvMRp5SFiJxUE6aU2eLo...,KR_7915137118,2,9,12
